In [4]:
import mediapipe
import cv2
import torch
import torchvision.models as models
import torch.nn as nn


In [5]:
# Paso 1: Reconstruir la arquitectura exactamente igual (ResNet18 con 29 clases)
model_nuevo = models.resnet18() # No necesitas poner weights aquí porque le vas a cargar los tuyos
num_ftrs = model_nuevo.fc.in_features
model_nuevo.fc = nn.Linear(num_ftrs, 29) # Reemplazamos la capa final para que coincida con tus 29 clases

# Paso 2: Cargar los pesos guardados en la estructura
ruta_guardado = "modelo_señas_resnet18.pt"
model_nuevo.load_state_dict(torch.load(ruta_guardado))

# Paso 3: ¡MUY IMPORTANTE! Poner el modelo en modo evaluación
model_nuevo.eval()

print("¡Modelo cargado y listo para usar en predicciones!")

¡Modelo cargado y listo para usar en predicciones!


In [7]:
import cv2
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# 1. CARGAR TU MODELO ENTRENADO
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18()
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 29) # Tus 29 clases

model.load_state_dict(torch.load("modelo_señas_resnet18.pt", map_location=device))
model.to(device)
model.eval() # Modo evaluación

# 2. DEFINIR LAS TRANSFORMACIONES (Exactamente las mismas del test)
transform_pipeline = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Lista de tus 29 clases en orden alfabético para mostrar el texto real en pantalla
# (Asegúrate de que coincida con el orden de tus carpetas de train)
clases_letras = ['A', 'B', 'C', 'D', 'DEL', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'NOTHING', 'O', 'P', 'Q', 'R', 'S', 'SPACE', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']

# 3. INICIAR CAPTURA DE PANTALLA / CÁMARA
cap = cv2.VideoCapture(0) # '0' es la cámara web por defecto

print("Presiona 'q' para salir...")

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    # OpenCV lee en BGR, lo convertimos a RGB para PyTorch
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Convertimos la matriz de OpenCV (NumPy) a una imagen PIL para aplicar los transforms
    imagen_pil = Image.fromarray(frame_rgb)
    
    # Aplicamos Resize, ToTensor y Normalize, y agregamos la dimensión del Batch [1, 3, 200, 200]
    tensor_img = transform_pipeline(imagen_pil).unsqueeze(0).to(device)
    
    # 4. PREDICCIÓN DEL MODELO
    with torch.no_grad():
        outputs = model(tensor_img)
        _, prediccion = torch.max(outputs, 1)
        indice_clase = prediccion.item()
        letra_detectada = clases_letras[indice_clase]
        
    # 5. MOSTRAR EL RESULTADO EN LA VENTANA DE OPENCV
    # Dibujamos el texto sobre el frame original
    cv2.putText(frame, f"Sena: {letra_detectada}", (30, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3, cv2.LINE_AA)
    
    # Mostramos el video en vivo
    cv2.imshow('Detector de Lenguaje de Senas', frame)
    
    # Si presionas la tecla 'q', el bucle se cierra
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Liberar la cámara y cerrar ventanas
cap.release()
cv2.destroyAllWindows()

Presiona 'q' para salir...


QFontDatabase: Cannot find font directory /home/fred/anaconda3/envs/pytorchenv/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/fred/anaconda3/envs/pytorchenv/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/fred/anaconda3/envs/pytorchenv/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/fred/anaconda3/envs/pytorchenv/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: C